# imports

In [110]:
import pandas as pd
from pathlib import Path

In [111]:
# cambiar la ruta a la carpeta raíz del proyecto
ROOT = Path("/Users/davidjaramillo/Documents/especializacion_bioinf_clinica/proyecto_final")
DATA_DIR = ROOT / "data"

path_mamografias = DATA_DIR / "mamografias_ML.xlsx"
path_patient_1 = DATA_DIR / "patient_IDs.xlsx"
path_patient_2 = DATA_DIR / "patient_IDs 2.xlsx"
path_birads = DATA_DIR / "imagenes_informes_BI-RADS.xlsx"
path_genomico = DATA_DIR / "genomico_sintetico.csv"

# pacientes

In [112]:

# 1. Cargar ambos archivos
df_patient_1 = pd.read_excel(path_patient_1)
# Si te da error de extensión o formato en el segundo, pandas lo leerá igual con read_excel
df_patient_2 = pd.read_excel(path_patient_2)

print("=== ARCHIVO 1: patient_IDs.xlsx ===")
print(f"Dimensiones: {df_patient_1.shape[0]} filas, {df_patient_1.shape[1]} columnas")
print("Columnas detectadas:", list(df_patient_1.columns))
print(f"Valores nulos por columna:\n{df_patient_1.isnull().sum()}\n")

print("=== ARCHIVO 2: patient_IDs 2.xlsx ===")
print(f"Dimensiones: {df_patient_2.shape[0]} filas, {df_patient_2.shape[1]} columnas")
print("Columnas detectadas:", list(df_patient_2.columns))
print(f"Valores nulos por columna:\n{df_patient_2.isnull().sum()}\n")

# 2. Comprobación rápida: ¿Son exactamente idénticos en su contenido?
son_identicos = df_patient_1.equals(df_patient_2)
print(f"¿Los dos archivos de IDs son exactamente idénticos?: {son_identicos}")

=== ARCHIVO 1: patient_IDs.xlsx ===
Dimensiones: 577 filas, 6 columnas
Columnas detectadas: ['patient', 'id_biop', 'visit_date', 'hospital', 'sex', 'birth_date']
Valores nulos por columna:
patient       1
id_biop       1
visit_date    1
hospital      1
sex           3
birth_date    1
dtype: int64

=== ARCHIVO 2: patient_IDs 2.xlsx ===
Dimensiones: 577 filas, 6 columnas
Columnas detectadas: ['patient', 'id_biop', 'visit_date', 'hospital', 'sex', 'birth_date']
Valores nulos por columna:
patient       1
id_biop       1
visit_date    1
hospital      1
sex           3
birth_date    1
dtype: int64

¿Los dos archivos de IDs son exactamente idénticos?: True


Descartamos el archivo patient_IDs 2.xlsx ya que es identico al patient_IDs.xlsx

la auditoría de nulos de este archivo maestro. Ambos tienen:

1 nulo en casi todas las columnas principales (patient, id_biop, visit_date, hospital, birth_date). Es muy probable que se trate de una fila completamente vacía al final del Excel o en medio.

3 nulos en la columna sex.
Investigamos filas corruptas

In [113]:
print("=== PACIENTES CON MÉTRICAS FALTANTES ===")
# Filtramos las filas donde 'diagnostic' es nulo O donde 'patient' es nulo
filas_na = df_patient_1[df_patient_1['patient'].isnull() | df_patient_1['sex'].isnull()]
print(filas_na)

# Ver cuántas veces aparece cada valor
print("\n=== CONTEO POR VALOR ===")
print(df_patient_1['sex'].value_counts(dropna=False))

=== PACIENTES CON MÉTRICAS FALTANTES ===
    patient   id_biop           visit_date                      hospital  sex  \
36      NaN       NaN                  NaN                           NaN  NaN   
438    p436  hxid4703  2020-12-28 00:00:00  Hospital Clínic de Barcelona  NaN   
555    p555  hxid4822  2023-04-17 00:00:00  Hospital Clínic de Barcelona  NaN   

              birth_date  
36                   NaN  
438  1978-05-13 00:00:00  
555  1979-12-21 00:00:00  

=== CONTEO POR VALOR ===
sex
Female    545
Male       29
NaN         3
Name: count, dtype: int64


In [114]:
df_patients_limpio = df_patient_1.copy()

# Eliminar filas donde todo sea nulo
df_patients_limpio = df_patients_limpio.dropna(how='all')

# Eliminar duplicados
df_patients_limpio = df_patients_limpio.drop_duplicates()

# Ordenar el dataframe para priorizar diagnósticos válidos y fechas más antiguas
df_patients_limpio = df_patients_limpio.sort_values(
    by=['patient', 'id_biop'], 
    ascending=[True, True], 
    na_position='last'
)

# Eliminar duplicados del ID de paciente reteniendo la primera aparición
df_patients_limpio = df_patients_limpio.drop_duplicates(subset=['patient', 'id_biop'], keep='first')

# Imputación de "sexo" con moda
moda_sexo = df_patients_limpio['sex'].mode()[0]
df_patients_limpio['sex'] = df_patients_limpio['sex'].fillna(moda_sexo)

# convertir a categoria hospital y sex
df_patients_limpio['hospital'] = df_patients_limpio['hospital'].astype('category')
df_patients_limpio['sex'] = df_patients_limpio['sex'].astype('category')

# Convertir las columnas de fechas a formato datetime
df_patients_limpio["visit_date"] = pd.to_datetime(df_patients_limpio["visit_date"])
df_patients_limpio["birth_date"] = pd.to_datetime(df_patients_limpio["birth_date"])

# Mostrar el conteo de valores nulos por columna
print("\n--- Conteo final de nulos por columna ---")
print(df_patients_limpio.info())


--- Conteo final de nulos por columna ---
<class 'pandas.DataFrame'>
Index: 569 entries, 0 to 101
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   patient     569 non-null    str           
 1   id_biop     569 non-null    str           
 2   visit_date  569 non-null    datetime64[ns]
 3   hospital    569 non-null    category      
 4   sex         569 non-null    category      
 5   birth_date  569 non-null    datetime64[ns]
dtypes: category(2), datetime64[ns](2), str(2)
memory usage: 23.4 KB
None


# mamografias

In [115]:
df_mamo = pd.read_excel(path_mamografias)

# 1. Mostrar las dimensiones del dataset (Filas, Columnas)
print("--- Dimensiones del dataset ---")
print(f"Filas: {df_mamo.shape[0]}, Columnas: {df_mamo.shape[1]}\n")

# 2. Ver las primeras filas para entender la estructura
print("--- Primeras 5 filas del dataset ---")
print(df_mamo.head())

# 3. Detectar valores nulos (vacíos) por cada columna
print("\n--- Conteo de valores nulos por columna ---")
print(df_mamo.isnull().sum())

# 4. Mostrar valores únicos por columna
print("\n--- Valores únicos ---")
print(df_mamo.nunique())
print("\n--- Información del DataFrame limpio ---")
print(df_mamo.info())

--- Dimensiones del dataset ---
Filas: 577, Columnas: 12

--- Primeras 5 filas del dataset ---
  patient diagnostic  radius  texture  perimeter    area  regularity  \
0      p1  Malignant   17.99    10.38     122.80  1001.0     0.11840   
1      p2  Malignant   20.57    17.77     132.90  1326.0     0.08474   
2      p3  Malignant   19.69    21.25     130.00  1203.0     0.10960   
3      p4  Malignant   11.42    20.38      77.58   386.1     0.14250   
4      p5        NaN   20.29    14.34     135.10  1297.0     0.10030   

   compactability  concavity  simetry  fractal_dimension       Fecha  
0         0.27760     0.3001   0.2419            0.07871  2022-06-16  
1         0.07864     0.0869   0.1812            0.05667  2022-04-02  
2         0.15990     0.1974   0.2069            0.05999  2022-04-20  
3         0.28390     0.2414   0.2597            0.09744  2022-02-06  
4         0.13280     0.1980   0.1809            0.05883  2022-12-29  

--- Conteo de valores nulos por columna ---
p

Paso 2: Investigar nulos encontrados

In [116]:
print("=== PACIENTES SIN DIAGNÓSTICO O CON MÉTRICAS FALTANTES ===")
# Filtramos las filas donde 'diagnostic' es nulo O donde 'patient' es nulo
filas_na = df_mamo[df_mamo['patient'].isnull() | df_mamo['diagnostic'].isnull()]
print(filas_na)

=== PACIENTES SIN DIAGNÓSTICO O CON MÉTRICAS FALTANTES ===
    patient diagnostic  radius  texture  perimeter    area  regularity  \
4        p5        NaN  20.290    14.34     135.10  1297.0     0.10030   
36      NaN        NaN     NaN      NaN        NaN     NaN         NaN   
49      p47        NaN   8.196    16.84      51.71   201.9     0.08600   
555    p555        NaN  12.880    28.92      82.50   514.3     0.08123   

     compactability  concavity  simetry  fractal_dimension       Fecha  
4           0.13280    0.19800   0.1809            0.05883  2022-12-29  
36              NaN        NaN      NaN                NaN         NaN  
49          0.05943    0.01588   0.1769            0.06503  2022-06-13  
555         0.05824    0.06195   0.1566            0.05708         NaN  


🔍 Hallazgos Clave de la Auditoría
La fila 36 está completamente vacía (NaN en todo): Es una fila corrupta o fantasma. No tiene ID de paciente (patient), ni diagnóstico, ni métricas. Acción: Hay que eliminarla por completo.

Pacientes p5, p47 y p555 no tienen Diagnóstico (diagnostic = NaN): * Las métricas de imagen de p5 y p47 son válidas y además tienen fecha en esta tabla.

La paciente p555 tampoco tiene diagnóstico ni fecha en esta tabla.

Acción: No las podemos borrar directamente porque perderíamos sus datos radiómicos. Más adelante intentaremos rescatar o cruzar su diagnóstico e información con los archivos de BI-RADS y IDs.

El problema masivo de la columna Fecha: Solo 101 filas de más de 550 tienen una fecha válida aquí. Las demás están vacías. Acción: Como dedujimos, la fuente primaria y fiable de las fechas serán los archivos patient_IDs.xlsx y patient_IDs 2.xlsx. Por ahora, eliminaremos esta columna corrupta de la tabla de mamografías para evitar duplicidades incorrectas en la base de datos final.

Escribir un código sencillo que solucione estos problemas detectados:

Eliminará las filas que estén completamente vacías (como la 36).

Eliminará la columna Fecha de este DataFrame (ya que la rescataremos de la tabla maestra de pacientes).

Guardará el resultado limpio en una nueva variable para que esté lista para la base de datos.

In [117]:
print(f"Filas antes de limpieza: {df_mamo.shape[0]}")
df_mamo_limpio = df_mamo.copy()

# 1. Eliminar filas donde todo sea nulo (como la fila 36)
df_mamo_limpio = df_mamo.dropna(how='all') # solo se borra si toda la fila está vacía

# 2. Eliminar duplicados
df_mamo_limpio = df_mamo_limpio.drop_duplicates()

print(f"Filas tras limpieza: {df_mamo_limpio.shape[0]}")

# # 4. Eliminar la columna 'Fecha' (está incompleta y la extraeremos de los archivos de IDs)
# df_mamo_limpio = df_mamo_limpio.drop(columns=['Fecha'])

print("\n--- Conteo de nulos en el nuevo dataset limpio ---")
print(df_mamo_limpio.info())

Filas antes de limpieza: 577
Filas tras limpieza: 571

--- Conteo de nulos en el nuevo dataset limpio ---
<class 'pandas.DataFrame'>
Index: 571 entries, 0 to 576
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   patient            571 non-null    str    
 1   diagnostic         568 non-null    str    
 2   radius             571 non-null    float64
 3   texture            571 non-null    float64
 4   perimeter          571 non-null    float64
 5   area               571 non-null    float64
 6   regularity         571 non-null    float64
 7   compactability     571 non-null    float64
 8   concavity          571 non-null    float64
 9   simetry            571 non-null    float64
 10  fractal_dimension  571 non-null    float64
 11  Fecha              100 non-null    str    
dtypes: float64(9), str(3)
memory usage: 58.0 KB
None


In [118]:
# show rows with duplicated patient IDs
print("\n=== Pacientes con IDs duplicados ===")
df_mamo_limpio[df_mamo_limpio.duplicated(subset=['patient'], keep=False)]


=== Pacientes con IDs duplicados ===


,patient,diagnostic,radius,texture,perimeter,area,regularity,compactability,concavity,simetry,fractal_dimension,Fecha
10,p11,Malignant,16.02,23.24,102.7,797.8,0.08206,0.06669,0.03299,0.1528,0.05697,2022-05-26
17,p11,Malignant,16.02,23.24,102.7,797.8,0.08206,0.06669,0.03299,0.1528,0.05697,2022-02-20
555,p555,NaN,12.88,28.92,82.5,514.3,0.08123,0.05824,0.06195,0.1566,0.05708,NaN
562,p555,Benign,12.88,28.92,82.5,514.3,0.08123,0.05824,0.06195,0.1566,0.05708,NaN


In [119]:
# Ver cuántas veces aparece cada valor
print("\n=== CONTEO POR VALOR ===")
print(df_mamo_limpio['diagnostic'].value_counts(dropna=False))


=== CONTEO POR VALOR ===
diagnostic
Benign       355
Malignant    211
NaN            3
Banign         1
Molignant      1
Name: count, dtype: int64


In [120]:
# Ordenar el dataframe para priorizar diagnósticos válidos y fechas más antiguas
df_mamo_limpio = df_mamo_limpio.sort_values(
    by=['patient', 'diagnostic', 'Fecha'], 
    ascending=[True, True, True], 
    na_position='last'
)

# Eliminar duplicados del ID de paciente reteniendo la primera aparición
df_mamo_limpio = df_mamo_limpio.drop_duplicates(subset=['patient'], keep='first')

# Corregir errores tipográficos en la columna 'diagnostic'
# 'Banign' → 'Benign'
# 'Molignant' → 'Malignant'

df_mamo_limpio['diagnostic'] = df_mamo_limpio['diagnostic'].replace({
    'Banign': 'Benign',
    'Molignant': 'Malignant'
})
# transformar fecha a datetime
df_mamo_limpio['Fecha'] = pd.to_datetime(df_mamo_limpio['Fecha'], errors='coerce')

print("\n=== VALORES DESPUÉS DE CORREGIR ===")
print(df_mamo_limpio.isnull().sum())


=== VALORES DESPUÉS DE CORREGIR ===
patient                0
diagnostic             2
radius                 0
texture                0
perimeter              0
area                   0
regularity             0
compactability         0
concavity              0
simetry                0
fractal_dimension      0
Fecha                470
dtype: int64


In [121]:
df_mamo_limpio.describe()

,radius,texture,perimeter,area,regularity,compactability,concavity,simetry,fractal_dimension,Fecha
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,99
mean,14.404673,19.289649,91.969033,662.023199,0.096360,0.104341,0.088799,0.184554,0.062798,2022-06-28 09:41:49.090909
min,6.981000,9.710000,43.790000,11.300000,0.052630,0.019380,0.000000,0.106000,0.049960,2022-01-03 00:00:00
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.161900,0.057700,2022-03-24 00:00:00
50%,13.380000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.179200,0.061540,2022-07-03 00:00:00
75%,16.030000,21.800000,104.100000,788.500000,0.105300,0.130400,0.130700,0.195900,0.066120,2022-09-22 12:00:00
max,102.400000,39.280000,188.500000,5021.000000,0.163400,0.345400,0.426800,1.300000,0.097440,2022-12-29 00:00:00
std,5.587400,4.301036,24.298981,397.433504,0.014064,0.052813,0.079720,0.064221,0.007060,NaN


# bi-rads

Este archivo contiene la categorización radiológica (BI-RADS), la cual mide el grado de sospecha de malignidad. Según la guía del proyecto, este tipo de archivos suele arrastrar problemas de formato (como espacios en blanco), inconsistencias en las categorías o filas vacías por errores de exportación.

In [122]:
df_birads = pd.read_excel(path_birads)

# 2. Mostrar las dimensiones originales
print("=== ARCHIVO: imagenes_informes_BI-RADS.xlsx ===")
print(f"Dimensiones originales: {df_birads.shape[0]} filas, {df_birads.shape[1]} columnas")

# 3. Detectar valores nulos por columna
print("\n--- Conteo inicial de nulos por columna ---")
print(df_birads.isnull().sum())

# Revisar los valores únicos de la columna de clasificación radiológica
print("\n=== CONTEO POR VALOR ===")
print(df_birads['Categoría BI-RADS'].value_counts(dropna=False))

=== ARCHIVO: imagenes_informes_BI-RADS.xlsx ===
Dimensiones originales: 577 filas, 5 columnas

--- Conteo inicial de nulos por columna ---
patient                       476
Hallazgo Imagen               477
Localización                  477
Categoría BI-RADS             477
Observaciones Radiológicas    477
dtype: int64

=== CONTEO POR VALOR ===
Categoría BI-RADS
NaN           477
BI-RADS 4A     23
BI-RADS 5      22
BI-RADS 4B     21
BI-RADS 4C     17
BI-RADS 3      17
Name: count, dtype: int64


El archivo original tiene 577 filas, pero 477 de ellas no tienen ni hallazgo, ni localización, ni BI-RADS.

Pacientes perdidos (476 nulos): Hay exactamente una fila que tiene el ID del paciente relleno (577 - 476 = 101 pacientes con ID), pero le faltan los datos clínicos, o viceversa.

Las categorías BI-RADS son limpias: Las filas que sí tienen datos utilizan una nomenclatura homogénea y estandarizada (BI-RADS 3, BI-RADS 4A, BI-RADS 4B, BI-RADS 4C, BI-RADS 5).

In [123]:
df_birads_limpio = df_birads.copy()

# Eliminar filas donde todo es nulo
df_birads_limpio = df_birads_limpio.dropna(how='all')
df_birads_limpio = df_birads_limpio.dropna(subset=['patient'])

# 3. Comprobar si queda algún nulo residual en el resto de columnas
print("\n--- Conteo final de nulos en BI-RADS limpio ---")
print(df_birads_limpio.isnull().sum())


--- Conteo final de nulos en BI-RADS limpio ---
patient                       0
Hallazgo Imagen               2
Localización                  2
Categoría BI-RADS             2
Observaciones Radiológicas    2
dtype: int64


Al limpiar el archivo de BI-RADS, hemos pasado de 577 filas a 101 filas reales. Esto significa que hemos eliminado de golpe 476 filas fantasma que eran puro ruido estructural en el Excel
El contraanálisis del conteo final: quedan 2 valores nulos en las columnas clínicas (Hallazgo Imagen, Localización, Categoría BI-RADS y Observaciones Radiológicas). 
Al tener patient = 0 nulos, significa que existen 2 pacientes reales que tienen su ID registrado pero cuyas celdas clínicas están completamente vacías.

Antes de tomar una decisión debemos ver quiénes son esos 2 pacientes y si coinciden con los problemas que arrastrábamos de las otras tablas.  

Paso 6: Identificar las inconsistencias remanentes en BI-RADS

In [124]:
# Mostrar las filas específicas donde 'Categoría BI-RADS' es nulo
print("=== PACIENTES CON DATOS CLÍNICOS VACÍOS EN BI-RADS ===")
pacientes_vacivos_birads = df_birads_limpio[df_birads_limpio['Categoría BI-RADS'].isnull()]
print(pacientes_vacivos_birads)

=== PACIENTES CON DATOS CLÍNICOS VACÍOS EN BI-RADS ===
    patient Hallazgo Imagen Localización Categoría BI-RADS  \
199    p197             NaN          NaN               NaN   
265    p263             NaN          NaN               NaN   

    Observaciones Radiológicas  
199                        NaN  
265                        NaN  


In [125]:
df_birads_limpio = df_birads_limpio.dropna(subset=['Hallazgo Imagen', 'Localización', 'Categoría BI-RADS',
       'Observaciones Radiológicas'])

"Checklist" de lo que llevamos auditado hasta ahora:
Para que veas cómo va cobrando forma tu memoria del proyecto, ya tenemos controladas 3 de las fuentes:

mamografias_ML: Limpio y sin nulos, salvo 3 diagnósticos pendientes (p5, p47, p555).

patient_IDs: 100% Limpio, descartado el archivo duplicado 2, corregido el sexo de p436 y p555.

imagenes_informes_BI-RADS: Reducido a las 101 filas reales, localizando ahora mismo los 2 últimos nulos.

Hemos identificado las dos filas culpables de los nulos residuales: las pacientes **p197 y p263**.

Lo que ha ocurrido aquí es que estas dos pacientes tienen abierto su registro en la base de datos de radiología (tienen ID asignado), pero el informe radiológico está completamente vacío (no se llegó a transcribir el hallazgo ni la categoría BI-RADS).

💡 Decisión metodológica 
Dado que el objetivo principal del proyecto es integrar datos multimodales para generar una nueva clasificación pronóstica basada en imagen y genómica, mantener filas donde las características de imagen son completamente inexistentes (NaN) no aporta valor y sesgaría los análisis estadísticos o algoritmos posteriores. Por tanto, la decisión correcta es eliminarlas de esta tabla específica y registrar la exclusión.

Con esto, podemos dar por finalizada la limpieza de los tres archivos clínicos y de imagen.

In [126]:
# df_mamo_limpio
print("\n--- Valores únicos ---")
print(df_mamo_limpio.nunique())
print("\n--- Información del DataFrame limpio ---")
print(df_mamo_limpio.info())

# mirar si hay duplicados en la columna patient
print("\n--- Valores únicos ---")
print(df_birads_limpio.nunique())
print("\n--- Información del DataFrame limpio ---")
print(df_birads_limpio.info())


--- Valores únicos ---
patient              569
diagnostic             2
radius               456
texture              479
perimeter            522
area                 540
regularity           474
compactability       537
concavity            537
simetry              432
fractal_dimension    499
Fecha                 82
dtype: int64

--- Información del DataFrame limpio ---
<class 'pandas.DataFrame'>
Index: 569 entries, 0 to 101
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   patient            569 non-null    str           
 1   diagnostic         567 non-null    str           
 2   radius             569 non-null    float64       
 3   texture            569 non-null    float64       
 4   perimeter          569 non-null    float64       
 5   area               569 non-null    float64       
 6   regularity         569 non-null    float64       
 7   compactability     569 non-null 

## preparar join

In [127]:
# mirar pasar esto a un script de qc
# para cada uno de los dfs mirar si hay pacientes que no están en df_patients_limpio
pacientes_patients = set(df_patients_limpio['patient'].dropna().unique())

for df, nombre in zip([df_mamo_limpio, df_birads_limpio], ['Mamografías', 'BI-RADS']):
    pacientes = set(df['patient'].dropna().unique())
    pacientes_solo_df = pacientes - pacientes_patients
    if len(pacientes_solo_df) > 0:
        print(f"Pacientes en {nombre} pero no en Patients: {len(pacientes_solo_df)}")
        print(pacientes_solo_df)

Pacientes en BI-RADS pero no en Patients: 1
{'p4745'}


# genomico

Este archivo tiene formato de variante genómica (estilo VCF estructurado en CSV). 
Las primeras columnas contienen metadatos de las mutaciones (#CHROM, POS, REF, ALT, etc.) y las siguientes columnas son las pacientes (p1, p2, p3...).

En bioinformática, los genotipos suelen venir codificados como:
0/0 : Homocigoto salvaje (sin mutación).
0/1 o 1/0 : Heterocigoto (presencia de la mutación en un alelo).
1/1 : Homocigoto mutado (presencia de la mutación en ambos alelos).
./. : Dato faltante o nulo (No Call).

Vamos a  detectar si hay datos nulos o formatos extraños.

In [128]:
# 1. Cargar el CSV
df_genomico = pd.read_csv(path_genomico, encoding='utf-8')

print("=== ARCHIVO GENÓMICO: genomico_sintetico.csv ===")
print(f"Dimensiones: {df_genomico.shape[0]} variantes (filas), {df_genomico.shape[1]} columnas (metadatos + pacientes)")

# 2. Inspeccionar las primeras columnas del df
columnas_seleccionadas = df_genomico.columns.tolist()[:20]
print("\n--- Ejemplo de las primeras variantes ---")
print(df_genomico[columnas_seleccionadas].head())

=== ARCHIVO GENÓMICO: genomico_sintetico.csv ===
Dimensiones: 2500 variantes (filas), 578 columnas (metadatos + pacientes)

--- Ejemplo de las primeras variantes ---
  #CHROM     POS      ID REF ALT  QUAL FILTER INFO FORMAT   p1   p2   p3   p4  \
0  chr21  731024  rs1000   A   T   100   PASS    .     GT  ./.  0/1  0/1  ./.   
1  chr18  955497  rs1001   G   A   100   PASS    .     GT  ./.  1/1  0/1  0/0   
2  chr17  440749  rs1002   C   G   100   PASS    .     GT  0/0  0/1  1/1  0/0   
3   chr6   96702  rs1003   C   T   100   PASS    .     GT  1/1  0/0  0/0  0/1   
4   chr6  244026  rs1004   A   G   100   PASS    .     GT  0/0  ./.  1/1  1/1   

    p5   p6   p7   p8   p9  p10  p11  
0  0/0  0/1  0/0  ./.  0/1  ./.  1/1  
1  ./.  0/0  0/1  0/0  0/0  1/1  0/1  
2  ./.  ./.  1/1  0/1  0/1  0/1  ./.  
3  0/1  ./.  1/1  0/0  0/0  1/1  0/0  
4  1/1  0/0  ./.  0/0  1/1  0/1  1/1  


Qué significan estos números tanto clínica como técnicamente:
Dimensiones del Dataset (2500 variantes $\times$ 578 columnas): 
* Las primeras 9 columnas corresponden a los metadatos estándar de un formato VCF (#CHROM, POS, ID, REF, ALT, QUAL, FILTER, INFO, FORMAT).
* Si restamos esas 9 columnas a las 578 totales, nos quedan exactamente 569 pacientes (578 - 9 = 569).
* Los Metadatos Genómicos están limpios: Todas las primeras filas muestran variantes bien estructuradas con calidad óptima (QUAL = 100) y han pasado los filtros de calidad (FILTER = PASS).
* El verdadero formato de los nulos (./.): Python no lee los datos faltantes como NaN porque la celda no está vacía; contiene el texto "./." (No Call). ¡Encontrar 603 "No Calls" solo en la paciente p1 demuestra que la matriz genómica tiene un alto grado de dispersión (sparsity)!

In [129]:
df_genomico_limpio = df_genomico.copy()
# Reemplazar los valores "./." por NaN para que pandas los reconozca como nulos
df_genomico_limpio.replace('./.', pd.NA, inplace=True)
# Reemplazar los valores "." por NaN en INFO
df_genomico_limpio.replace('.', pd.NA, inplace=True)

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,p1,...,p560,p561,p562,p563,p564,p565,p566,p567,p568,p569
0,chr21,731024,rs1000,A,T,100,PASS,NaN,GT,NaN,...,1/1,0/0,1/1,0/1,NaN,NaN,0/1,1/1,0/0,1/1
1,chr18,955497,rs1001,G,A,100,PASS,NaN,GT,NaN,...,0/0,1/1,1/1,NaN,0/1,0/0,0/0,1/1,NaN,NaN
2,chr17,440749,rs1002,C,G,100,PASS,NaN,GT,0/0,...,1/1,0/0,0/0,0/0,NaN,NaN,NaN,NaN,0/0,0/0
3,chr6,96702,rs1003,C,T,100,PASS,NaN,GT,1/1,...,0/1,1/1,0/1,NaN,1/1,1/1,0/0,1/1,0/0,0/0
4,chr6,244026,rs1004,A,G,100,PASS,NaN,GT,0/0,...,NaN,0/0,NaN,0/0,0/0,NaN,0/1,0/0,NaN,1/1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,chr22,496897,rs3495,G,C,100,PASS,NaN,GT,0/0,...,1/1,1/1,0/0,NaN,1/1,NaN,0/1,NaN,0/1,1/1
2496,chr14,775710,rs3496,T,G,100,PASS,NaN,GT,NaN,...,0/1,1/1,0/1,NaN,0/0,NaN,0/0,0/0,1/1,0/0
2497,chr12,514114,rs3497,C,A,100,PASS,NaN,GT,0/1,...,0/1,NaN,0/0,NaN,1/1,NaN,NaN,1/1,0/1,1/1
2498,chr21,381103,rs3498,C,T,100,PASS,NaN,GT,0/0,...,1/1,1/1,0/0,0/1,0/1,0/0,0/1,1/1,NaN,0/0


In [130]:
# melt para convertir de formato ancho a largo
df_genomico_largo = df_genomico_limpio.melt(
    id_vars=[col for col in df_genomico_limpio.columns if not col.startswith('p')],  # mantener las columnas de metadatos
    value_vars=[col for col in df_genomico_limpio.columns if col.startswith('p')],
    var_name='PATIENT',
    value_name='GENOTYPE'
)
print(df_genomico_largo.head())

  #CHROM     POS      ID REF ALT  QUAL FILTER INFO FORMAT PATIENT GENOTYPE
0  chr21  731024  rs1000   A   T   100   PASS  NaN     GT      p1      NaN
1  chr18  955497  rs1001   G   A   100   PASS  NaN     GT      p1      NaN
2  chr17  440749  rs1002   C   G   100   PASS  NaN     GT      p1      0/0
3   chr6   96702  rs1003   C   T   100   PASS  NaN     GT      p1      1/1
4   chr6  244026  rs1004   A   G   100   PASS  NaN     GT      p1      0/0


In [131]:
df_genomico_largo.nunique()

#CHROM        22
POS         2494
ID          2500
REF            4
ALT            4
QUAL           1
FILTER         1
INFO           0
FORMAT         1
PATIENT      569
GENOTYPE       3
dtype: int64

In [132]:
# drop cols que no aportan información relevante para el análisis
df_genomico_largo.drop(columns=['QUAL', 'FILTER', 'INFO', 'FORMAT'], inplace=True)

# drop filas donde GENOTYPE es nulo
df_genomico_largo.dropna(subset=['GENOTYPE'], inplace=True)

# convertir a categoría
cols_to_category = ['REF', 'ALT', 'FILTER', 'FORMAT', 'GENOTYPE']
for col in cols_to_category:
    if col in df_genomico_largo.columns:
        df_genomico_largo[col] = df_genomico_largo[col].astype('category')

In [133]:
df_genomico_largo.info()

<class 'pandas.DataFrame'>
Index: 1067662 entries, 2 to 1422498
Data columns (total 7 columns):
 #   Column    Non-Null Count    Dtype   
---  ------    --------------    -----   
 0   #CHROM    1067662 non-null  str     
 1   POS       1067662 non-null  int64   
 2   ID        1067662 non-null  str     
 3   REF       1067662 non-null  category
 4   ALT       1067662 non-null  category
 5   PATIENT   1067662 non-null  str     
 6   GENOTYPE  1067662 non-null  category
dtypes: category(3), int64(1), str(3)
memory usage: 43.8 MB


# Arquitectura SQL

In [134]:
import pandas as pd
from pathlib import Path
import sqlite3

Crear DDBB

In [135]:
# 1. Definir rutas relativas estructuradas
# cambiar la ruta a la carpeta raíz del proyecto
ROOT = Path("/Users/davidjaramillo/Documents/especializacion_bioinf_clinica/proyecto_final")
DB_PATH = ROOT / "proyecto_bic.db"
SCHEMA_PATH = ROOT / "scripts" / "schema.sql"

# 2. Conectar al motor (crea el archivo .db automáticamente si no existe)
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# 3. Forzar de inmediato la integridad referencial (Claves Foráneas)
cursor.execute("PRAGMA foreign_keys = ON;")

# 4. Leer el archivo SQL externo
with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
    schema_sql = f.read()

# 5. Ejecutar todas las sentencias DDL (CREATE TABLE, restricciones, etc.)
try:
    cursor.executescript(schema_sql)
    conn.commit()
    print(f"Éxito: Base de datos creada en: {DB_PATH.resolve()}")
    print("Esquema relacional inyectado correctamente.")
except sqlite3.Error as e:
    conn.rollback()
    print(f"Error crítico al ejecutar el esquema SQL: {e}")
finally:
    # Garantizar el cierre de la conexión para evitar bloqueos del archivo de BD
    conn.close()

Éxito: Base de datos creada en: /Users/davidjaramillo/Documents/especializacion_bioinf_clinica/proyecto_final/proyecto_bic.db
Esquema relacional inyectado correctamente.


renombrar variables

In [136]:
# rename columns
df_birads_limpio.rename(columns={
    'Hallazgo Imagen': 'hallazgo',
    'Localización': 'localizacion',
    'Categoría BI-RADS': 'birads_cat',
    'Observaciones Radiológicas': 'observaciones'
}, inplace=True)

df_genomico_largo.rename(columns={
    '#CHROM': 'chrom',
    'POS': 'pos',
    'ID': 'variant_id',
    'REF': 'ref',
    'ALT': 'alt',
    'FILTER': 'filter',
    'FORMAT': 'format',
    'PATIENT': 'patient',
    'GENOTYPE': 'genotype',
}, inplace=True)

verificar ids en patients

In [138]:
# Lista maestra estricta
pacientes_validos = set(df_patients_limpio['patient'].dropna())

print(df_birads_limpio['patient'].isin(pacientes_validos).all())
print(df_genomico_largo['patient'].isin(pacientes_validos).all())
print(df_mamo_limpio['patient'].isin(pacientes_validos).all())


False
True
True


In [139]:
# Purgar huérfanos en memoria
df_birads_limpio = df_birads_limpio[df_birads_limpio['patient'].isin(pacientes_validos)]

Cargar datos

In [140]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

try:
    # Activar el soporte de claves foráneas en la sesión actual de la conexión
    cursor.execute("PRAGMA foreign_keys = ON;")
    
    # Inserción masiva indexada
    df_patients_limpio.to_sql(
        name='pacientes',          # Nombre exacto de la tabla en el schema.sql
        con=conn,                  # Objeto de conexión nativo de sqlite3
        if_exists='append',        # Inserta los datos sin alterar la estructura existente
        index=False                # Evita que Pandas cree una columna con el índice del DataFrame
    )

    df_mamo_limpio.to_sql(
        name='mamografias',
        con=conn,
        if_exists='append',
        index=False
    )

    df_birads_limpio.to_sql(
        name='hallazgos_radiologicos',
        con=conn,
        if_exists='append',
        index=False
    )

    df_genomico_largo.to_sql(
        name='variantes_genomicas',
        con=conn,
        if_exists='append',
        index=False
    )
    
    # Confirmar la transacción en el almacenamiento físico
    conn.commit()
    print(f"Carga exitosa")

except sqlite3.IntegrityError as e:
    # Captura violaciones de clave primaria o restricciones UNIQUE de forma controlada
    conn.rollback()
    print(f"Error crítico de integridad al insertar: {e}")

finally:
    # Cerrar la conexión para liberar el archivo .db y evitar el error 'database is locked'
    conn.close()

Carga exitosa
